# **Irrigation needs extraction tool**
<font color="#FBB800">Agricultural Cold Chain Analysis and Prioritization (AgCAP) Tool</font><br>

-------
# Description
This notebook enriches an AgCAP settlement point dataset with **irrigation need indicators** derived from the [TerraClimate](https://www.climatologylab.org/terraclimate.html) global climatological dataset, accessed through **Google Earth Engine (GEE)**.

It is a notebook version of `scripts/aux_scripts/irrigation/irrigation_extraction_MOZ.py` (see `README_irrigation_extraction.md` in that folder for background) and performs the same operations:
1) Build a monthly evapotranspiration-stress classification pipeline in GEE from TerraClimate data over a chosen study period
2) Load the AgCAP settlement point dataset (`.fgb`, `.gpkg`, or `.csv`)
3) Extract, per settlement, how many months per year fall into each irrigation-need category (chunked GEE requests)
4) Merge the results back onto the settlements and export the enriched dataset

Output columns added:

| Output column | Meaning |
|---|---|
| `Irrigation: months with no need (rainfed)` | Months where rainfall alone is sufficient |
| `Irrigation: months with (moderate) need` | Months where supplemental irrigation is beneficial |
| `Irrigation: months with (critical) need` | Months where irrigation is essential for crop production |
| `Irrigation: months with (moderate + critical) need` | Combined moderate + critical months |

Values range from 0 to 12 (months per year).

-------
# License and Copyright Notice

This notebook is part of the **AgCAP** project.

**Copyright (C) 2025 Sustainable Energy for All**

This program is free software: you can redistribute it and/or modify it under the terms of the **GNU Affero General Public License version 3 (AGPLv3)** as published by the Free Software Foundation.

This work is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU Affero General Public License for more details.

You should have received a copy of the GNU Affero General Public License along with this program. If not, see <https://www.gnu.org/licenses/agpl-3.0.html>.

**[Optional: Add a link to source code archive for AGPL compliance]**
*Download Source Code for this Version:* [Link to GitHub Tag Archive]

In [ ]:
# Copyright (C) 2025 Sustainable Energy for All
#
#This program is free software: you can redistribute it and/or modify it 
# under the terms of the **GNU Affero General Public License version 3 (AGPLv3)** 
# as published by the Free Software Foundation.
#
# See <https://www.gnu.org/licenses/agpl-3.0.html>.

# Preparation

## Import packages and functions

In [ ]:
import os
import sys
import warnings
from pathlib import Path
from datetime import datetime

import ee
import geemap
import pandas as pd
import geopandas as gpd

warnings.filterwarnings("ignore")

# Load the autoreload extension
%load_ext autoreload
%autoreload 2

current_dir = Path.cwd()
project_root = current_dir.parent

# Insert the project root path into the system path
# This allows the notebook to find the 'scripts' folder as a package.
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

## Constants
Column renames applied to the extracted GEE output, and the supported output file formats/drivers. These match the standalone script exactly.

In [ ]:
COLUMN_RENAMES = {
    "rainfed_months":      "Irrigation: months with no need (rainfed)",
    "supplemental_months": "Irrigation: months with (moderate) need",
    "severe_months":       "Irrigation: months with (critical) need",
    "red_yellow_months":   "Irrigation: months with (moderate + critical) need",
}

OUTPUT_EXTENSIONS = {"fgb": ".fgb", "gpkg": ".gpkg", "csv": ".csv"}
OUTPUT_DRIVERS    = {"fgb": "FlatGeobuf", "gpkg": "GPKG"}

## Configuration
Set the input/output paths and output format here.

Accepted input formats: FlatGeobuf (`.fgb`), GeoPackage (`.gpkg`), or CSV (`.csv`, must contain a `geometry` WKT column or `lon`/`lat`/`longitude`/`latitude` columns).

In [ ]:
# --- Input: AgCAP settlement dataset to enrich ---
INPUT_PATH = Path(project_root / "data/processed/input_analyzed/settlements_analyzed_20260720.gpkg")  # <-- update this to your AgCAP input file

# --- Output: folder, file name (without extension), and format ---
OUTPUT_DIR    = INPUT_PATH.parent
OUTPUT_NAME   = INPUT_PATH.stem + "_irrigation"
OUTPUT_FORMAT = "gpkg"  # options: "fgb", "gpkg", "csv"

assert OUTPUT_FORMAT in OUTPUT_EXTENSIONS, f"OUTPUT_FORMAT must be one of {list(OUTPUT_EXTENSIONS)}"

print(f"Input file    : {INPUT_PATH}")
print(f"Output folder : {OUTPUT_DIR}")
print(f"Output file   : {OUTPUT_NAME}{OUTPUT_EXTENSIONS[OUTPUT_FORMAT]}")

## Google Earth Engine project & study period
You need a free GEE account and a linked Google Cloud project — sign up at https://earthengine.google.com/signup and find your project ID at https://code.earthengine.google.com/ (top-left dropdown, e.g. `ee-yourname`).

The tool computes a long-term monthly climatology over the study period below. A longer period produces more stable averages. TerraClimate data is available from 1958 to the present.

In [ ]:
GEE_PROJECT = "gee-project-name"  # <-- update this to your GEE Cloud project ID

START_YEAR = 2004  # first year of the climatological period (inclusive), min 1958
END_YEAR   = 2024  # last year of the climatological period (inclusive), must be >= START_YEAR

assert 1958 <= START_YEAR <= END_YEAR, "START_YEAR must be >= 1958 and <= END_YEAR"

## Advanced parameters
These control the agronomic classification and the GEE extraction behaviour — the defaults work well in most cases.

In [ ]:
# Months where mean temperature falls below this threshold (°C) are classified as
# 'dormant' (growing season inactive) regardless of water availability.
# Recommended: 5 °C for most tropical and sub-tropical regions.
BASE_TEMP = 5.0

# Spatial resolution (metres) used when sampling TerraClimate data in GEE.
# TerraClimate's native resolution is ~4,000 m. Using a lower value does NOT add
# detail but greatly increases computation time.
SCALE = 4000

# Number of points sent to GEE per request. Larger chunks mean fewer round-trips
# but higher memory use. Reduce (e.g. 250 or 500) if you hit payload/timeout errors.
CHUNK_SIZE = 1000

## Connect to Google Earth Engine
Tries to initialise silently first; if that fails, runs the browser-based authentication flow. Credentials are saved locally, so future runs authenticate automatically.

In [ ]:
def initialize_gee(project_id):
    try:
        ee.Initialize(project=project_id)
        print(f"Connected to GEE project: {project_id}")
        return
    except Exception:
        pass

    print("Could not authenticate automatically. Starting authentication flow.")
    print("A browser window will open — sign in with the Google account linked to")
    print("your GEE account and grant the requested access, then return here.")

    ee.Authenticate()

    try:
        ee.Initialize(project=project_id)
        print(f"Authentication successful. Connected to: {project_id}")
    except Exception as exc:
        print("Initialisation failed after authentication.")
        print(f"Error: {exc}")
        print("Possible causes: misspelled/nonexistent project ID, no access to the")
        print("project, or signed in with a different Google account than the one")
        print("registered with GEE.")
        raise


initialize_gee(GEE_PROJECT)

## Build the Earth Engine processing pipeline
For each calendar month, computes the mean TerraClimate PET/AET/temperature over the study period, derives the evapotranspiration stress ratio (AET/PET), and classifies each pixel as critical / moderate / rainfed / dormant. Sums the classification across the 12 months to get, per pixel, how many months per year fall into each category.

In [ ]:
def build_extraction_image(start_year, end_year, base_temp):
    terraclimate = (
        ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")
        .filterDate(f"{start_year}-01-01", f"{end_year + 1}-01-01")
    )

    def process_month(m):
        m = ee.Number(m)
        mean_img = terraclimate.filter(
            ee.Filter.calendarRange(m, m, "month")
        ).mean()

        pr_raw = mean_img.select("pr")
        coastal_mask = pr_raw.mask().focalMax(1, "square", "pixels")

        def scaled_band(name):
            raw = mean_img.select(name).multiply(0.1)
            return raw.unmask(raw.focalMean(1, "square", "pixels")).updateMask(coastal_mask)

        pet  = scaled_band("pet")
        aet  = scaled_band("aet")
        tmmx = scaled_band("tmmx")
        tmmn = scaled_band("tmmn")

        tmean        = tmmx.add(tmmn).divide(2)
        safe_pet     = pet.where(pet.eq(0), 0.001)
        stress_ratio = aet.divide(safe_pet)

        fao_class = (
            ee.Image(0)
            .where(stress_ratio.gte(0.5), 1)
            .where(stress_ratio.gte(0.8), 2)
            .where(tmean.lt(base_temp), 3)
            .updateMask(coastal_mask)
            .rename("fao_class")
        )
        return fao_class.set("month", m)

    months  = ee.List.sequence(1, 12)
    monthly = ee.ImageCollection.fromImages(months.map(process_month))

    severe_months  = monthly.map(lambda img: img.eq(0)).sum().rename("severe_months")
    supp_months    = monthly.map(lambda img: img.eq(1)).sum().rename("supplemental_months")
    rainfed_months = monthly.map(lambda img: img.eq(2)).sum().rename("rainfed_months")
    red_yellow     = severe_months.add(supp_months).rename("red_yellow_months")

    return ee.Image([rainfed_months, supp_months, severe_months, red_yellow]).toInt()


extraction_image = build_extraction_image(START_YEAR, END_YEAR, BASE_TEMP)
print("Pipeline ready.")

## Load the input settlement data
Reads `.fgb`/`.gpkg` via `pyogrio`, or `.csv` (with a `geometry` WKT column, or `lon`/`lat`/`longitude`/`latitude` columns), and reprojects to EPSG:4326 if needed.

In [ ]:
def load_vector(path):
    path = str(path)
    ext = os.path.splitext(path)[1].lower()

    if ext in (".fgb", ".gpkg"):
        gdf = gpd.read_file(path, engine="pyogrio")
    elif ext == ".csv":
        df = pd.read_csv(path)
        if "geometry" in df.columns:
            import shapely.wkt
            df["geometry"] = df["geometry"].apply(shapely.wkt.loads)
            gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")
        elif {"lon", "lat"}.issubset(df.columns):
            gdf = gpd.GeoDataFrame(
                df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326"
            )
        elif {"longitude", "latitude"}.issubset(df.columns):
            gdf = gpd.GeoDataFrame(
                df,
                geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
                crs="EPSG:4326",
            )
        else:
            raise ValueError(
                "CSV has no recognised geometry column. "
                "Expected: 'geometry' (WKT), or 'lon'/'lat', or 'longitude'/'latitude'."
            )
    else:
        raise ValueError(f"Unsupported file format: {ext}. Expected: .fgb, .gpkg, or .csv")

    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs("EPSG:4326")

    return gdf


gdf = load_vector(INPUT_PATH)
print(f"Loaded {len(gdf):,} features.")
gdf.head()

## Extract values from Google Earth Engine (chunked)
Sends the input points to GEE in batches of `CHUNK_SIZE`, running `reduceRegions` with a mean reducer at `SCALE` metres. Larger chunks mean fewer round-trips but higher memory use — reduce `CHUNK_SIZE` above if you hit payload or timeout errors.

In [ ]:
def extract_chunked(extraction_image, gdf, chunk_size, scale):
    gdf = gdf.copy()
    gdf["_tid"] = range(len(gdf))

    total    = len(gdf)
    n_chunks = (total + chunk_size - 1) // chunk_size
    print(f"Total points : {total:,}")
    print(f"Chunk size   : {chunk_size:,}  ->  {n_chunks} request(s) to GEE")

    results = []
    for i in range(0, total, chunk_size):
        chunk_num = i // chunk_size + 1
        chunk     = gdf.iloc[i : i + chunk_size][["_tid", "geometry"]]
        end_idx   = min(i + chunk_size, total)
        print(f"[{chunk_num:>3}/{n_chunks}] Points {i + 1:,}-{end_idx:,}...", end=" ", flush=True)

        ee_fc = geemap.geopandas_to_ee(chunk)
        extracted = extraction_image.reduceRegions(
            collection=ee_fc,
            reducer=ee.Reducer.mean(),
            scale=scale,
            tileScale=4,
        )

        try:
            features = extracted.getInfo()["features"]
            rows = [f["properties"] for f in features]
            results.append(pd.DataFrame(rows))
            print("done")
        except Exception as exc:
            print(f"error: {exc}")

    return results, gdf


results, gdf = extract_chunked(extraction_image, gdf, CHUNK_SIZE, SCALE)

if not results:
    raise RuntimeError("No data was extracted. Please check the errors above.")

## Merge results
Concatenates the extracted chunks, renames the columns to their descriptive labels, and merges them back onto the original settlement dataset.

In [ ]:
extracted_stats = pd.concat(results, ignore_index=True).rename(columns=COLUMN_RENAMES)
final_gdf = gdf.merge(extracted_stats, on="_tid", how="left").drop(columns=["_tid"])
print(f"Merged {len(final_gdf):,} records.")
final_gdf.head()

## Save output
Saves to `OUTPUT_DIR / OUTPUT_NAME` in the chosen `OUTPUT_FORMAT` (FlatGeobuf/GeoPackage retain geometry; CSV drops it).

In [ ]:
def save_output(final_gdf, output_dir, output_name, output_format):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    ext      = OUTPUT_EXTENSIONS[output_format]
    out_path = output_dir / (output_name + ext)

    if output_format == "csv":
        df = final_gdf.drop(columns=["geometry"], errors="ignore")
        df.to_csv(out_path, index=False)
    else:
        final_gdf.to_file(out_path, driver=OUTPUT_DRIVERS[output_format], engine="pyogrio")

    return out_path


out_path = save_output(final_gdf, OUTPUT_DIR, OUTPUT_NAME, OUTPUT_FORMAT)
print(f"Saved to: {out_path}")
print("Extraction complete.")